# Silver To Gold
Runs the KPI functions from `app/metrics.py` on silver and builds one row per player for the app.

In [ ]:
import sys
from pathlib import Path

import yaml
from pyspark.sql import SparkSession, functions as F

sys.path.append("..")                      # project root, so utility can import app.metrics
from utility import layer_path, build_gold

spark = SparkSession.builder.master("local[*]").appName("silver_to_gold").getOrCreate()

# Parameters

In [ ]:
cfg = yaml.safe_load(open("../config/keyplayer_battle.yaml", encoding="utf-8"))

BASE = Path("..") / cfg["data"]["root"]
YEAR, GAME = cfg["active_game"]["year"], cfg["active_game"]["game"]
BLC = cfg["events"]["blueline_crossing"]
print(f"Processing {YEAR}/{GAME:02d}, fast crossing ≥ {BLC['fast_threshold']} m/s, {BLC['direction']}")

# Join KPIs 

In [ ]:
silver = lambda kind: spark.read.parquet(str(layer_path(cfg, BASE, "silver", kind, YEAR, GAME)))

gold = build_gold(silver("stats"), silver("events"), silver("players"),
                  BLC["fast_threshold"], BLC["direction"], YEAR, GAME,
                  name_fixes=cfg.get("name_corrections"))


# Checks

In [ ]:
n = gold.count()
assert gold.select("player_id").distinct().count() == n, "Duplicate player_id in gold"
print(n, "players:", {r.role: r["count"] for r in gold.groupBy("role").count().collect()})

gold.filter("role = 'goalie'").select("name", "Team", "SA", "SV_pct", "GA", "xGa").show()
gold.filter("role = 'skater'").orderBy(F.desc("xGsum")).select(
    "name", "Team", "xGsum", "S", "Shots_on_target_pct", "Fast_blueline_crossings", "Pass_partners").show(5)
unknown = [r["name"] for r in gold.filter(F.col("role") == "unknown").select("name").collect()]
print("Unknown roles:", unknown or "none")

# Save

In [ ]:
out = layer_path(cfg, BASE, "gold", "player_metrics", YEAR, GAME)
gold.write.mode("overwrite").parquet(str(out))
print(f"gold: {gold.count()} rows → {out}")

In [ ]:
gold.select("name", "`TOI (s)`", "`Dist (m)`").show(5)